# Linear Probe: Does LLaVA Encode Depth Ordering?

We extract frozen patch tokens from LLaVA's visual encoder (SigLIP), then train a logistic
regression to predict **pairwise depth ordering** (which of two patches is closer to the camera).
DepthAnything v2 depth values serve as pseudo-ground-truth.

We compare:
- **LLaVA patch tokens** (layer 16 of SigLIP ViT-L/14@384)
- **DepthAnything v2 features** (its own intermediate features)

If LLaVA patch tokens predict depth ordering near chance (50%) while DepthAnything features
score >80%, the representation bottleneck hypothesis is confirmed.

**Drive layout:** same as baseline notebook.
**Runtime:** A100 GPU recommended.

In [1]:
# ── 1. Install ───────────────────────────────────────────────────────────────
!pip install -q transformers>=4.45.0 accelerate>=0.27.0 pillow tqdm scikit-learn

In [2]:
# ── 2. Mount Drive ────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH  = "/content/drive/MyDrive/MindCube/data/raw/MindCube_tinybench.jsonl"
IMAGE_ROOT = "/content/drive/MyDrive/MindCube/data/"
LLAVA_ID   = "/content/drive/MyDrive/models/llava-onevision-qwen2-7b-ov-hf"

import pathlib
assert pathlib.Path(DATA_PATH).exists()
print("Drive mounted.")

Mounted at /content/drive
Drive mounted.


In [3]:
# ── 3. Load DepthAnything v2 ──────────────────────────────────────────────────
import torch
import numpy as np
from PIL import Image
from transformers import pipeline, AutoImageProcessor, AutoModelForDepthEstimation

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

DA_MODEL_ID = "depth-anything/Depth-Anything-V2-Small-hf"

depth_pipe = pipeline(
    task="depth-estimation",
    model=DA_MODEL_ID,
    device=0 if device == "cuda" else -1,
)

# Also load the model directly so we can hook intermediate features
da_processor = AutoImageProcessor.from_pretrained(DA_MODEL_ID)
da_model = AutoModelForDepthEstimation.from_pretrained(
    DA_MODEL_ID, torch_dtype=torch.float32
).to(device).eval()

print("DepthAnything v2 loaded.")
print(f"DA model class: {da_model.__class__.__name__}")

Device: cuda


config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/99.2M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

The image processor of type `DPTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

DepthAnything v2 loaded.
DA model class: DepthAnythingForDepthEstimation


In [4]:
# ── 4. Load LLaVA visual encoder (SigLIP) ────────────────────────────────────
import gc
from transformers import LlavaOnevisionForConditionalGeneration, AutoProcessor

print(f"Loading LLaVA from {LLAVA_ID} ...")
llava_processor = AutoProcessor.from_pretrained(LLAVA_ID)
llava_model = LlavaOnevisionForConditionalGeneration.from_pretrained(
    LLAVA_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    attn_implementation="sdpa",
)
llava_model.eval()
llava_processor.image_processor.do_image_splitting = False

# The vision tower lives at .model.vision_tower in the HF format
vision_tower = llava_model.model.vision_tower
gc.collect()
torch.cuda.empty_cache()
print(f"LLaVA loaded. Vision tower: {vision_tower.__class__.__name__}")
n_layers = len(vision_tower.vision_model.encoder.layers)
print(f"Number of encoder layers: {n_layers}")

Loading LLaVA from /content/drive/MyDrive/models/llava-onevision-qwen2-7b-ov-hf ...


The image processor of type `LlavaOnevisionImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/765 [00:00<?, ?it/s]

LLaVA loaded. Vision tower: SiglipVisionModel
Number of encoder layers: 26


In [5]:
# ── 5. Feature extraction helpers ────────────────────────────────────────────
import json
from pathlib import Path
from tqdm.notebook import tqdm

PROBE_LAYER = 16  # middle of SigLIP's 27 layers


@torch.inference_mode()
def extract_llava_patch_features(pil_image: Image.Image) -> np.ndarray:
    """
    Returns patch features from SigLIP layer PROBE_LAYER.
    Shape: (N_patches, hidden_dim)  e.g. (729, 1152)
    """
    raw = llava_processor.image_processor(images=[pil_image], return_tensors="pt")
    pixel_values = raw.pixel_values
    # LlavaOnevisionImageProcessor returns a list of tensors (one per image)
    if isinstance(pixel_values, list):
        pixel_values = pixel_values[0]
    # Collapse any extra leading dims until shape is (B, C, H, W)
    while pixel_values.dim() > 4:
        pixel_values = pixel_values.squeeze(0)
    if pixel_values.dim() == 3:
        pixel_values = pixel_values.unsqueeze(0)
    pixel_values = pixel_values.to(vision_tower.device).to(torch.float16)

    out = vision_tower(pixel_values=pixel_values, output_hidden_states=True)
    # hidden_states[0] = patch embeddings, [1..n] = after each encoder layer
    feats = out.hidden_states[PROBE_LAYER + 1]  # (1, N_patches, D)
    return feats[0].float().cpu().numpy()  # (N_patches, D)


@torch.inference_mode()
def get_depth_map_array(pil_image: Image.Image) -> np.ndarray:
    """Run DepthAnything, return (H, W) float32 depth map."""
    out = depth_pipe(pil_image)
    return np.array(out["depth"], dtype=np.float32)


@torch.inference_mode()
def extract_da_patch_features(pil_image: Image.Image) -> np.ndarray:
    """
    Extract DepthAnything v2 backbone patch features (DINOv2-small).
    Returns (N_spatial_tokens, hidden_dim) after stripping the CLS token.
    """
    inputs = da_processor(images=pil_image, return_tensors="pt").to(device)
    out = da_model.backbone(**inputs, output_hidden_states=True, return_dict=True)
    last_hs = out.hidden_states[-1]            # (1, N_tokens, D)
    tokens = last_hs[0].float().cpu().numpy()  # (N_tokens, D)
    # DINOv2 prepends a CLS token — strip it if N is not a perfect square
    n = tokens.shape[0]
    if int(n ** 0.5) ** 2 != n:
        tokens = tokens[1:]
    return tokens  # (N_spatial, D)


# Quick smoke test
_test = Image.new("RGB", (384, 384), color=(100, 150, 200))
_lf = extract_llava_patch_features(_test)
_df = extract_da_patch_features(_test)
print(f"LLaVA features shape: {_lf.shape}")
print(f"DA features shape:    {_df.shape}")
print("Feature extraction helpers ready.")

LLaVA features shape: (729, 1152)
DA features shape:    (1369, 384)
Feature extraction helpers ready.


In [6]:
# ── 5b. Sanity check on 3 real MindCube images ───────────────────────────────
import json
from pathlib import Path

# Grab the first 3 unique image paths from the dataset
_smoke_paths = []
with open(DATA_PATH) as _f:
    for _line in _f:
        _rec = json.loads(_line)
        for _rel in _rec["images"]:
            _p = str(Path(IMAGE_ROOT) / _rel)
            if _p not in _smoke_paths:
                _smoke_paths.append(_p)
            if len(_smoke_paths) >= 3:
                break

print("Testing feature extraction on 3 real images...")
for _path in _smoke_paths:
    _img = Image.open(_path).convert("RGB")
    _depth = get_depth_map_array(_img)
    _lf = extract_llava_patch_features(_img)
    _df = extract_da_patch_features(_img)
    _img.close()
    print(f"  {Path(_path).name}: depth={_depth.shape}  llava={_lf.shape}  da={_df.shape}")

print("Sanity check passed — proceed to full run.")

Testing feature extraction on 3 real images...
  front_007.jpg: depth=(640, 480)  llava=(729, 1152)  da=(1813, 384)
  left_084.jpg: depth=(640, 480)  llava=(729, 1152)  da=(1813, 384)
  back_157.jpg: depth=(640, 480)  llava=(729, 1152)  da=(1813, 384)
  front_022.jpg: depth=(640, 480)  llava=(729, 1152)  da=(1813, 384)
  left_172.jpg: depth=(640, 480)  llava=(729, 1152)  da=(1813, 384)
  front_268.jpg: depth=(640, 480)  llava=(729, 1152)  da=(1813, 384)
  front_138.png: depth=(270, 480)  llava=(729, 1152)  da=(777, 384)
  front_350.jpg: depth=(640, 480)  llava=(729, 1152)  da=(1813, 384)
  front_036.jpg: depth=(640, 480)  llava=(729, 1152)  da=(1813, 384)
  front_020.jpg: depth=(640, 480)  llava=(729, 1152)  da=(1813, 384)


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  front_150.png: depth=(270, 480)  llava=(729, 1152)  da=(777, 384)
  front_008.jpg: depth=(640, 480)  llava=(729, 1152)  da=(1813, 384)
  right_249.png: depth=(270, 480)  llava=(729, 1152)  da=(777, 384)
  left_103.jpg: depth=(640, 480)  llava=(729, 1152)  da=(1813, 384)
  front_006.jpg: depth=(640, 480)  llava=(729, 1152)  da=(1813, 384)
  front_027.jpg: depth=(640, 480)  llava=(729, 1152)  da=(1813, 384)
  front_125.png: depth=(270, 480)  llava=(729, 1152)  da=(777, 384)
  front_339.jpg: depth=(640, 480)  llava=(729, 1152)  da=(1813, 384)
  front_231.jpg: depth=(640, 480)  llava=(729, 1152)  da=(1813, 384)
  right_188.png: depth=(270, 480)  llava=(729, 1152)  da=(777, 384)
  front_242.jpg: depth=(640, 480)  llava=(729, 1152)  da=(1813, 384)
  front_013.jpg: depth=(640, 480)  llava=(729, 1152)  da=(1813, 384)
  front_241.jpg: depth=(640, 480)  llava=(729, 1152)  da=(1813, 384)
  back_133.jpg: depth=(640, 480)  llava=(729, 1152)  da=(1813, 384)
  front_116.png: depth=(270, 480)  llava

In [7]:
# ── 6. Build pairwise depth-ordering dataset ──────────────────────────────────
#
# For each MindCube image:
#   1. Get depth map from DepthAnything v2
#   2. Extract LLaVA patch features + DA patch features
#   3. Sample PAIRS_PER_IMAGE random pairs of patches
#   4. Label: 1 if patch_i is closer (higher depth value) than patch_j, else 0
#   5. Feature for pair: concatenate (feat_i - feat_j) as the contrast feature
#
# We do this for the first MAX_IMAGES images (to keep wall-clock < 30 min).

MAX_IMAGES      = 300   # use ~300 images from MindCube
PAIRS_PER_IMAGE = 200   # pairs per image → 60k total training pairs
TRAIN_FRAC      = 0.8

rng = np.random.default_rng(42)

# Load image list from dataset
image_paths = []
with open(DATA_PATH) as f:
    for line in f:
        rec = json.loads(line)
        for rel_path in rec["images"]:
            abs_path = str(Path(IMAGE_ROOT) / rel_path)
            image_paths.append(abs_path)
            if len(image_paths) >= MAX_IMAGES:
                break
    if len(image_paths) < MAX_IMAGES:
        pass  # dataset has fewer images, that's fine

image_paths = list(dict.fromkeys(image_paths))[:MAX_IMAGES]  # deduplicate
print(f"Processing {len(image_paths)} images...")

llava_X, llava_y = [], []
da_X,    da_y    = [], []

for img_path in tqdm(image_paths):
    try:
        img = Image.open(img_path).convert("RGB")

        depth_arr = get_depth_map_array(img)      # (H, W)
        llava_feats = extract_llava_patch_features(img)  # (729, 1152)
        da_feats    = extract_da_patch_features(img)     # (N, D)

        img.close()

        # Map depth to patch grid for LLaVA (27x27 patches)
        H, W = depth_arr.shape
        n_llava = llava_feats.shape[0]
        grid = int(n_llava ** 0.5)
        depth_resized = np.array(
            Image.fromarray(depth_arr).resize((grid, grid), Image.BILINEAR)
        ).flatten()  # (729,)

        # Map depth to DA patch grid
        n_da = da_feats.shape[0]
        da_grid = int(n_da ** 0.5)
        depth_da = np.array(
            Image.fromarray(depth_arr).resize((da_grid, da_grid), Image.BILINEAR)
        ).flatten()  # (N_da,)

        # Sample pairs for LLaVA features
        idxA = rng.integers(0, n_llava, size=PAIRS_PER_IMAGE)
        idxB = rng.integers(0, n_llava, size=PAIRS_PER_IMAGE)
        valid = idxA != idxB
        idxA, idxB = idxA[valid], idxB[valid]
        diff = llava_feats[idxA] - llava_feats[idxB]  # (K, D)
        labels = (depth_resized[idxA] > depth_resized[idxB]).astype(int)
        llava_X.append(diff)
        llava_y.append(labels)

        # Sample pairs for DA features
        idxA2 = rng.integers(0, n_da, size=PAIRS_PER_IMAGE)
        idxB2 = rng.integers(0, n_da, size=PAIRS_PER_IMAGE)
        valid2 = idxA2 != idxB2
        idxA2, idxB2 = idxA2[valid2], idxB2[valid2]
        diff2 = da_feats[idxA2] - da_feats[idxB2]
        labels2 = (depth_da[idxA2] > depth_da[idxB2]).astype(int)
        da_X.append(diff2)
        da_y.append(labels2)

    except Exception as e:
        print(f"[WARN] {img_path}: {e}")
        continue

llava_X = np.vstack(llava_X).astype(np.float32)
llava_y = np.concatenate(llava_y)
da_X    = np.vstack(da_X).astype(np.float32)
da_y    = np.concatenate(da_y)

print(f"LLaVA pairs: {llava_X.shape[0]:,}  |  DA pairs: {da_X.shape[0]:,}")
print(f"LLaVA feature dim: {llava_X.shape[1]}  |  DA feature dim: {da_X.shape[1]}")

Processing 270 images...


  0%|          | 0/270 [00:00<?, ?it/s]

[WARN] /content/drive/MyDrive/MindCube/data/other_all_image/among/shoe_216/front_007.jpg: index 1809 is out of bounds for axis 0 with size 1764
[WARN] /content/drive/MyDrive/MindCube/data/other_all_image/among/shoe_216/left_084.jpg: index 1785 is out of bounds for axis 0 with size 1764
[WARN] /content/drive/MyDrive/MindCube/data/other_all_image/among/shoe_216/back_157.jpg: index 1787 is out of bounds for axis 0 with size 1764
[WARN] /content/drive/MyDrive/MindCube/data/other_all_image/among/shoe_216/right_246.jpg: index 1793 is out of bounds for axis 0 with size 1764
[WARN] /content/drive/MyDrive/MindCube/data/other_all_image/among/bottle_211/front_022.jpg: index 1805 is out of bounds for axis 0 with size 1764
[WARN] /content/drive/MyDrive/MindCube/data/other_all_image/among/bottle_211/left_083.jpg: index 1812 is out of bounds for axis 0 with size 1764
[WARN] /content/drive/MyDrive/MindCube/data/other_all_image/among/bottle_211/back_157.jpg: index 1801 is out of bounds for axis 0 with 

In [8]:
# ── 7. Train & evaluate linear probes ────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score


def probe(X, y, label, train_frac=0.8):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, train_size=train_frac, random_state=42, stratify=y
    )
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)
    X_te = scaler.transform(X_te)

    clf = LogisticRegression(
        max_iter=500, C=1.0, solver="lbfgs", n_jobs=-1
    )
    clf.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, clf.predict(X_te))
    print(f"[{label}]  test accuracy = {acc:.4f}  "
          f"(train={len(y_tr):,}, test={len(y_te):,})")
    return acc


print("Training probes...")
llava_acc = probe(llava_X, llava_y, f"LLaVA-OV patch tokens (layer {PROBE_LAYER})")
da_acc    = probe(da_X,    da_y,    "DepthAnything v2 features")

print(f"\n{'='*50}")
print(f"  Depth ordering probe (chance = 0.500)")
print(f"  LLaVA-OV layer {PROBE_LAYER:2d} features: {llava_acc:.4f}")
print(f"  DepthAnything v2 features:  {da_acc:.4f}")
print(f"  Gap (DA - LLaVA):           {da_acc - llava_acc:+.4f}")
print(f"{'='*50}")

probe_results = {
    "llava_layer": PROBE_LAYER,
    "n_images": len(image_paths),
    "pairs_per_image": PAIRS_PER_IMAGE,
    "llava_probe_accuracy": llava_acc,
    "da_probe_accuracy": da_acc,
    "chance": 0.5,
}
with open("/content/probe_results.json", "w") as f:
    import json
    json.dump(probe_results, f, indent=2)
print("Probe results saved to /content/probe_results.json")

Training probes...
[LLaVA-OV patch tokens (layer 16)]  test accuracy = 0.8041  (train=43,135, test=10,784)
[DepthAnything v2 features]  test accuracy = 0.5950  (train=799, test=200)

  Depth ordering probe (chance = 0.500)
  LLaVA-OV layer 16 features: 0.8041
  DepthAnything v2 features:  0.5950
  Gap (DA - LLaVA):           -0.2091
Probe results saved to /content/probe_results.json


In [9]:
# ── 8. Download results ───────────────────────────────────────────────────────
from google.colab import files
files.download("/content/probe_results.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>